# 1. PDF 파일 준비 및 실행환경 구성

In [ ]:
'''
https://www.bok.or.kr/portal/bbs/P0002359/view.do?nttId=10096683&searchCnd=1&searchKwd=&depth2=200699&depth3=200066&depth=200066&pageUnit=10&pageIndex=1&programType=newsData&menuNo=200066&oldMenuNo=200066
'''

## 설치 패키지
- pip install llama-cloud
- pip install llama-index
- pip install llama-index-vector-stores-pinecone    --- # pinecone 호환
- pip install llama-index-embeddings-openai         --- # openai 호환 : 같이 설치될 수도           
** 혹시 추가 설치하는 게 있을 수도 ...

In [ ]:

# $ pip list | findstr llama
# llama_cloud                              2.3.0
# llama-index                              0.14.20
# llama-index-core                         0.14.20
# llama-index-embeddings-openai            0.6.0
# llama-index-instrumentation              0.5.0
# llama-index-llms-openai                  0.7.5
# llama-index-vector-stores-pinecone       0.8.0
# llama-index-workflows                    2.18.0

## LLAMA CLOUD 계정 등록
- 홈페이지 : https://cloud.llamaindex.ai/    
- 계정 등록 및 API 키 생성 : LLAMA_CLOUD_API_KEY

In [5]:
from dotenv import load_dotenv
import os

# 1. 환경 변수 로드 및 설정
load_dotenv(r"D:\Practice\hipython\llm\.env", override=True)
LLAMA_PARSE_API_KEY = os.environ['LLAMA_CLOUD_API_KEY']
PINECONE_API_KEY = os.environ['PINECONE_API_KEY']

# 2. PDF 문서 로드
PDF_PATH = "./data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"


# 2. 주피터 노트북용 : LLAMA-CLOUD 버전

## ✅ 1. PDF 파일 로딩 및 페이지 구조화

In [49]:
from llama_cloud import LlamaCloud, AsyncLlamaCloud

# 2. 함수 정의 : PDF 파일 업
def load_and_parse(file_path):
    # 최신 동기 클라이언트 초기화
    client = LlamaCloud(api_key=LLAMA_PARSE_API_KEY)

    print(f"문서 업로드 및 파싱 시작: {os.path.basename(file_path)}")

    # 3. PDF 업로드 및 파싱 요청
    with open(file_path, "rb") as f:
        # purpose="parse" : 파싱 목적
        file_obj = client.files.create(file=f, purpose="parse")
    
    # 파일 구조 파싱 - Table은 마크다운 형식으로
    result = client.parsing.parse(
        file_id=file_obj.id,
        tier="agentic",
        version="latest",
        
        # 테이블 및 이미지(화면) 추출
        output_options={
            "markdown": {
                "tables": {
                    "output_tables_as_markdown": True,  # 
                },
            },
            "images_to_save": ["screenshot"],
        },

        # 이미지 -> TEXT 추출 OCR 언어 : 한글, 영어
        processing_options={
            "ocr_parameters": {
                "languages": ["ko", "en"]  #
            }
        },

        expand=["markdown", "items", "images_content_metadata"],
    )
    
    return result


result    
 ├─ markdown.pages        
 ├─ text.pages           
 ├─ items (table 등)             
 └─ images           

## ✅ 2. Document 생성 및 분할

In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.node_parser import MarkdownNodeParser

def result_to_nodes(result):

    # 1. Document 생성 - 페이지 단위
    documents = []

    for page in result.markdown.pages:
        if page.markdown.strip():
            documents.append(
                Document(
                    text=page.markdown,
                    metadata={
                        "page"  : page.page_number,  # 페이지 번호는 1부터 시작
                        'title' : '2026년 2월 경제전망보고서',
                    }  # 마크다운 페이지 포함 metadata 내 포함 필요없음
                )
            )
    
    # # 2. splitter 정의 
    # splitter = SentenceSplitter(
    #     chunk_size=500,
    #     chunk_overlap=100
    # )

    # # 3. documnet chunking 
    # nodes = splitter.get_nodes_from_documents(documents)
    
    # 🔥 2. Markdown 구조 기반 파싱 : 표 분할 방지
    parser = MarkdownNodeParser(
        chunk_size=800,  # GPT 추천
        chunk_overlap=100
    )

    nodes = parser.get_nodes_from_documents(documents)

    return nodes

## ✅ 3. Pinecone Indexing & Upsert

In [51]:
from pinecone import Pinecone, ServerlessSpec
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import StorageContext
# from langchain_openai import OpenAIEmbeddings

def build_pinecone_index(nodes, api_key, index_name, namespace):

    # 1. Pinecone client 생성
    pc = Pinecone(api_key=api_key)
    
    # 2. embedding (🔥 LlamaIndex 방식)
    # ✅ 반드시 먼저 설정
    Settings.embed_model = OpenAIEmbedding(
        model="text-embedding-3-small"
    )
    # embed_model = OpenAIEmbedding(model="text-embedding-3-small")
    # Settings.embed_model = embed_model

    # 2. index 목록 조회
    existing_indexes = [idx.name for idx in pc.list_indexes()]

    # 3. index 없으면 생성
    if index_name not in existing_indexes:
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

    # 4. index 연결
    pinecone_index = pc.Index(index_name)

    # 5. LlamaIndex vector store
    vector_store = PineconeVectorStore(
        pinecone_index=pinecone_index,
        namespace=namespace   # 🔥 중요
    )

    # 🔥 6. nodes 기반 indexing (핵심 변경)
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store
    )

    index = VectorStoreIndex(
        nodes=nodes,
        storage_context=storage_context
    )

    # # 2. 명시적 insert
    # index.insert_nodes(nodes)
    
    return index, pinecone_index

## ✅ 4. 문서 로딩 및 파이콘 업서트 실행

In [52]:
# 1. parsing
result = load_and_parse(PDF_PATH)

문서 업로드 및 파싱 시작: 2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf


In [56]:
# 2. document 변환
nodes = result_to_nodes(result)

In [57]:
nodes

[TextNode(id_='a35ff8c6-f6a2-4b9b-b21d-0784da752bfb', embedding=None, metadata={'page': 1, 'title': '2026년 2월 경제전망보고서', 'header_path': '/'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='409449ab-09bd-4b05-a253-d4d4f90cfbf8', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page': 1, 'title': '2026년 2월 경제전망보고서'}, hash='d75486ef4ff753ee878c798e1ba34fb5cb0e6ffee5784881ddb902b128898e51'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='414c38f4-18e8-465a-932a-551bb93e4d3f', node_type=<ObjectType.TEXT: '1'>, metadata={'header_path': '/'}, hash='4a666830bdcafe651771c002a1a33604186b1cd8ec4c934537d71c71ca09a6ee')}, metadata_template='{key}: {value}', metadata_separator='\n', text='![경제전망 로고](page_1_image_2_v2.jpg)', mimetype='text/plain', start_char_idx=0, end_char_idx=33, text_template='{metadata_str}\n\n{content}'),
 TextNode(id_='414c38f4-18e8-465a-932a-551bb93e4d3f', embedding=None, metadata={

In [58]:
# 3. index 생성
INDEX_NAME = "finance-bok-table"
NAMESPACE  = "bok-ns-t"

index, pinecone_index = build_pinecone_index(nodes, api_key=os.environ["PINECONE_API_KEY"], index_name=INDEX_NAME, namespace=NAMESPACE)

Upserted vectors:   0%|          | 0/223 [00:00<?, ?it/s]

## ✅ 5. Pinecone Upsert 확인

In [59]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns-t': {'vector_count': 223}},
 'total_vector_count': 223,
 'vector_type': 'dense'}


# 3. RAG 파이프라인 구성 및 테스트
## 3.1. retriever 생성 및 검색 테스트

In [39]:
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding

# 설정
INDEX_NAME = "finance-bok-table"
NAMESPACE = "bok-ns-t"

# 1. embedding 설정 (필수)
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# 2. Pinecone 연결
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
pinecone_index = pc.Index(INDEX_NAME)

# 3. vector store 생성
vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    namespace=NAMESPACE
)

# 4. index 생성 (이미 저장된 벡터 기반)
index = VectorStoreIndex.from_vector_store(vector_store)

# 5. query engine 생성
query_engine = index.as_query_engine(similarity_top_k=3)

# 6. 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "고용 시장 전망은?"
]

for query in queries:
    print(f"[ 질문 ] {query}")
    
    response = query_engine.query(query)

    # 🔥 source_nodes에서 실제 chunk 확인
    for i, node in enumerate(response.source_nodes):
        text = node.node.text[:100]
        page = node.node.metadata.get("page", "")
        print(f"  {i+1}. (p.{page}) {text}")

    print('-'*100)

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.84) GDP 성장률>


| 연도   | 전망오차(%p) |
| ---- | -------- |
| 2000 | 2.5      |
| 2001 | -3.8     |
| 2002 | 
  2. (p.13) **GDP 성장률<sup>1)</sup>**
| 전망시점   | 최솟값 | 하위 25% | 중윗값 | 상위 25% | 최댓값 |
| ------ | --- | ------ | --
  3. (p.37) # 2. 거시경제 전망

## 경제성장

**2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 개선세 확대, 예상보다 양호한 세계경제 흐름 등
----------------------------------------------------------------------------------------------------
[ 질문 ] 물가 상승률 전망은?
  1. (p.86) 5            | -0.4             | -0.7             | -1.0             | -1.7             | -1.4     
  2. (p.55) | 전망시점   | 최솟값 | 하위 25% | 중윗값 | 상위 25% | 최댓값 |
| ------ | --- | ------ | --- | ------ | --- |
| 25.5
  3. (p.49) [그림 2.24] 소비자물가 상승률 전망
(전년동기대비, %)
| Category | 25.11월 전망 | 26.2월 전망 |
| -------- | --------- | ----
----------------------------------------------------------------------------------------------------
[ 질문 ] 고용 시장 전망은?
  1. (p.12) # □ 올해 경상수지 흑자규모는 지난 전망경로를 크게 상회하는 1,700억달러로 예상

## 3.2. RAG 체인 구성

In [60]:
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.prompts import PromptTemplate

# 설정
INDEX_NAME = "finance-bok-table"
NAMESPACE = "bok-ns-t"

# 1. 모델 설정
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

# 2. Pinecone 연결
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
pinecone_index = pc.Index(INDEX_NAME)

vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    namespace=NAMESPACE
)    
    
# 3. index 생성
index = VectorStoreIndex.from_vector_store(vector_store)

# 4. Prompt 정의
qa_prompt = PromptTemplate("""
    당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
    아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
    답변 시 참조한 문서의 페이지 번호들을 답변 끝에 추가하고
    문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

    [참고문서]
    {context_str}

    [질문]
    {query_str}

    한글로 간결하고 정확하게 답변하세요.
""")

# 5. Query Engine 생성
query_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=qa_prompt
)

# 6. RAG 테스트
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")
    response = query_engine.query(q)
    print(f"[ A ] {response}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다. (페이지 7)

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ A ] 2026년 소비자물가 상승률은 2.0%로 전망되며, 이는 지난해 11월의 2.1% 전망보다 소폭 하향 조정된 수치입니다. 2026년 상반기에는 2.1%, 하반기에는 2.2%로 예상됩니다. 전반적으로 전년 동기 대비 상승률은 2.0%에 근접할 것으로 보입니다. (페이지 11, 48)

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 2026년 상반기 수출 전망은 3,951억 달러로, 전년 동기 대비 18.1% 증가할 것으로 예상됩니다. 2026년 하반기에는 4,001억 달러로 6.8% 증가할 것으로 보이며, 2026년 연간 수출은 7,952억 달러로 12.1% 증가할 것으로 전망됩니다. (페이지 46)



# 4. RAG 파이프라인 동작 확인 및 답변 품질 평가
## 4.1. 검색 품질 확인

In [41]:
queries = [
    "2026년 경제성장률 전망은?",
    "소비자물가 상승률 전망은?",
    "수출 전망은 어떻게 되나요?"
]

# 🔥 retriever 생성
retriever = index.as_retriever(similarity_top_k=3)

for query in queries:
    print(f"[ 질문 ] {query}")
    
    results = retriever.retrieve(query)

    print(f"검색된 청크 수: {len(results)}")
    
    for i, r in enumerate(results):
        text = r.node.text[:120]
        page = r.node.metadata.get("page", "")
        print(f"  {i+1}. (p.{page}) {text}")
    
    print()

[ 질문 ] 2026년 경제성장률 전망은?
검색된 청크 수: 3
  1. (p.84) GDP 성장률>


| 연도   | 전망오차(%p) |
| ---- | -------- |
| 2000 | 2.5      |
| 2001 | -3.8     |
| 2002 | 2.2      |
| 2003 | 
  2. (p.13) **GDP 성장률<sup>1)</sup>**
| 전망시점   | 최솟값 | 하위 25% | 중윗값 | 상위 25% | 최댓값 |
| ------ | --- | ------ | --- | ------ | --- |
|
  3. (p.37) # 2. 거시경제 전망

## 경제성장

**2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 개선세 확대, 예상보다 양호한 세계경제 흐름 등에 힘입어 성장률이 지난해<sub>1

[ 질문 ] 소비자물가 상승률 전망은?
검색된 청크 수: 3
  1. (p.86) 5            | -0.4             | -0.7             | -1.0             | -1.7             | -1.4             | 95% 신뢰구간 하
  2. (p.13) **소비자물가 상승률<sup>2)</sup>**
| 전망시점   | 최솟값 | 하위 25% | 중윗값 | 상위 25% | 최댓값 |
| ------ | --- | ------ | --- | ------ | --- |
  3. (p.84) <2. 소비자물가 상승률<sup>1)</sup>>

| 연도   | 전망오차(%p) |
| ---- | -------- |
| 2000 | -0.5     |
| 2001 | 1.2      |
| 2002 | -0

[ 질문 ] 수출 전망은 어떻게 되나요?
검색된 청크 수: 3
  1. (p.37) | 구분   | 내수  | 수출   |
| ---- | --- | ---- |
| 24.상 | 2.5 | 0.3  |
| 24.하 | 1.2 | 0.

## 4.2. 답변 품질 확인

In [61]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

# LLM 설정
llm_base = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.llm = llm_base

questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")

    # 🔹 일반 LLM (RAG 없음)
    base_response = llm_base.complete(q)
    print(f"[ 일반 LLM ] {str(base_response)[:150]}")

    # 🔹 RAG 기반 LLM
    rag_response = query_engine.query(q)
    print(f"[ RAG 답변 ] {str(rag_response)[:150]}")

    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있습니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다. 

정확한 수치를 원하신다면, 국제통화기금(IMF), 세계은행, 또는 각국의 중앙
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다. (페이지 7)

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ 일반 LLM ] 소비자물가 상승률 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 통화 정책, 원자재 가격, 공급망 문제, 그리고 글로벌 경제 상황 등이 주요한 영향을 미칩니다. 

2023년의 경우, 많은 국가들이 인플레이션 압력을 경험하고 있으며, 이는 에너
[ RAG 답변 ] 2026년 소비자물가 상승률은 2.0%로 전망되며, 이는 지난해 11월의 2.1% 전망보다 소폭 하향 조정된 수치입니다. 2026년 상반기에는 2.1%, 하반기에는 2.2%로 예상됩니다. 전반적으로 수요측 압력이 제한적이지만, 전자기기 및 일부 서비스의 가격 인상 압력

[ Q ] 수출 전망은 어떻게 되나요?
[ 일반 LLM ] 수출 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 글로벌 수요, 환율 변동, 무역 정책, 그리고 특정 산업의 경쟁력 등이 중요한 요소로 작용합니다. 

2023년의 경우, 세계 경제의 회복세와 함께 일부 국가에서는 수출이 증가할 것으로 예상되
[ RAG 답변 ] 2026년 상반기 수출 전망은 3,951억 달러로, 전년 동기 대비 18.1% 증가할 것으로 예상됩니다. 2026년 하반기에는 4,001억 달러로 6.8% 증가할 것으로 보이며, 2026년 연간 수출은 7,952억 달러로 12.1% 증가할 것으로 전망됩니다. (페이지 



## 4.3. 종합 품질 점검

In [62]:
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]

In [63]:
for q in test_questions:
    print(f"[ Q ] {q}")

    # 🔹 일반 LLM (RAG 없음)
    base_response = llm_base.complete(q)
    print(f"[ 일반 LLM ] {str(base_response)[:150]}")

    # 🔹 RAG 기반 LLM
    rag_response = query_engine.query(q)
    print(f"[ RAG 답변 ] {str(rag_response)[:150]}")

    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년의 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있으며, 특정한 수치를 제공하기 위해서는 최신 경제 보고서나 예측 자료를 참조해야 합니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다.
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다. (페이지 7)

[ Q ] 2026년 소비자물가 상승률 전망치는?
[ 일반 LLM ] 2026년 소비자물가 상승률에 대한 구체적인 전망치는 여러 경제 기관이나 정부의 경제 전망에 따라 달라질 수 있습니다. 일반적으로 이러한 전망치는 경제 성장, 통화 정책, 국제 유가, 공급망 문제 등 다양한 요인에 의해 영향을 받습니다. 

정확한 수치를 알고 싶다면,
[ RAG 답변 ] 2026년 소비자물가 상승률 전망치는 중윗값 기준으로 2.0%입니다. (페이지 12, 48)

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
[ 일반 LLM ] 2026년의 수출 전망 금액에 대한 구체적인 수치는 여러 요인에 따라 달라질 수 있으며, 각국의 경제 상황, 글로벌 시장 동향, 무역 정책 등에 영향을 받습니다. 따라서 정확한 수출 전망 금액을 제공하기는 어렵습니다. 

각국의 정부 기관이나 경제 연구소에서 발표하는 
[ RAG 답변 ] 2026년 수출 전망 금액은 7,952억 달러입니다. (페이지 46)

[ Q ] 2030년 경제성장률 전망은?
[ 일반 LLM ] 2030년의 경제성장률 전망은 여러 요인에 따라 달라질 수 있습니다. 각국의 정부, 국제기구(예: IMF, 세계은행), 그리고 경제 연구 기관들이 다양한 경제 지표와 트렌드를 분석하여 예측을 제공합니다. 

일반적으로 경제성장률은 다음과 같은 요소에 영향을 받습니다:

[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
[ 일반 LLM ] 미국 연방준비제도

## 4.4. 청킹 전략 비교 실습 (제외)

In [ ]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# # 전략 A — 작은 청크 (정밀 검색)
# splitter_a = RecursiveCharacterTextSplitter(
#     chunk_size=200,
#     chunk_overlap=20
# )

# # 전략 B — 중간 청크 (권장)
# splitter_b = RecursiveCharacterTextSplitter(
#     chunk_size=500,
#     chunk_overlap=50
# )

# # 전략 C — 큰 청크 (문맥 유지)
# splitter_c = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=100
# )

# for name, splitter in [("A(200)", splitter_a), ("B(500)", splitter_b), ("C(1000)", splitter_c)]:
#     splits = splitter.split_documents(docs)
#     print(f"전략 {name}: 청크 수 {len(splits)}개, 평균 길이 {sum(len(s.page_content) for s in splits)//len(splits)}자")

- 향후 구성

[문서]        
   ↓          
Llama Parse       
   ↓            
Llama Index      
   ↓             
FAISS (로컬)   <= 파인콘 대신 RAG 인덱스 저장 활용               
   ↓                    
Query Engine       